In [1]:
%%capture
!pip install gliner[training]

In [ ]:
# !pip uninstall sympy -y
# !pip install sympy==1.12

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [38]:
import json
import random

train_path = "/content/drive/MyDrive/dataset/gliner/train.jsonl"
test_path = "/content/drive/MyDrive/dataset/gliner/test.jsonl"
val_path = "/content/drive/MyDrive/dataset/gliner/valid.jsonl"

def load_jsonl_data(filepath):
    data = []
    with open(filepath, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

train_data = load_jsonl_data(train_path)
test_data = load_jsonl_data(test_path)
val_data = load_jsonl_data(val_path)


print('Dataset is splitted...')
print("Train size:", len(train_data))
print("Validation size:", len(val_data))
print("Test size:", len(test_data))

Dataset is splitted...
Train size: 22749
Validation size: 2138
Test size: 3104


In [39]:
test_data[100]

{'tokenized_text': ['त्यांनी',
  'सिद्धू',
  'आणि',
  'घई',
  'यांना',
  'सीटच्या',
  'खाली',
  'आणि',
  'किट',
  'बॅगच्या',
  'मागे',
  'लपवलं',
  'होतं'],
 'ner': [[1, 1, 'person'], [3, 3, 'person']]}

In [52]:
from gliner import GLiNER
trainer = None
# Load a pretrained model
model = GLiNER.from_pretrained("urchade/gliner_multi")

# Train the model
trainer = model.train_model(
    train_dataset=train_data,
    eval_dataset=val_data,
    output_dir="/content/drive/MyDrive/dataset/marathi-ner",

    max_steps=1000,
    #save_steps=500,
    warmup_ratio=0.15,

    lr_scheduler_type="cosine",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    others_lr=1e-4,
    weight_decay=0.01,
    others_weight_decay=0.01,
    max_grad_norm=1.0,

    focal_loss_alpha=0.25,
    focal_loss_gamma=2.0,
    focal_loss_prob_margin=0.0,
    loss_reduction="sum",   # KEY CHANGE
    negatives=0.5,
    masking="none",
)

# Save the trained model
trainer.save_model()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 0}.


Step,Training Loss
10,21.277679
20,21.293301
30,6.229908
40,6.703181
50,5.731450
60,5.034715
70,4.228159
80,4.006899
90,3.708882
100,3.737315


In [53]:
# Save the trained model
trainer.save_model("/content/drive/MyDrive/dataset/marathi-ner")

In [46]:
metrics = trainer.evaluate(test_data)
print(metrics)

{'eval_loss': 3.1188414096832275, 'eval_runtime': 29.8935, 'eval_samples_per_second': 103.835, 'eval_steps_per_second': 12.979, 'epoch': 0.10548523206751055}


In [54]:
from gliner import GLiNER
#urchade/gliner_multi_pii-v1
# Load from local directory
model = GLiNER.from_pretrained("/content/drive/MyDrive/dataset/marathi-ner")

In [57]:
text = "नवी दिल्ली अभिनेता सुशांतसिंह राजपूत मृत्यू प्रकरणाचा तपास आता सीबीआयच्या ताब्यात आहे"
# Define labels for entity extraction
labels = ["person", "location", "organization", "measure", "time", "date", "designation"]

# Perform entity prediction
entities = model.predict_entities(text, labels, threshold=0.1)

# Display predicted entities and their labels
for entity in entities:
    print(entity["text"], "=>", entity["label"])

नव => date
द => person
ल => person
ी => person
अभ => person
ह र => person
जप => person
प => person
रकरण => person
तप => person


In [50]:
entities

[{'start': 20,
  'end': 22,
  'text': '३०',
  'label': 'measure',
  'score': 0.7391752600669861},
 {'start': 49,
  'end': 51,
  'text': '१९',
  'label': 'measure',
  'score': 0.7676753997802734}]

In [ ]:
!cp -r "/content/sample_data/pii" "/content/drive/MyDrive/Pre-Train-PII"